In [ ]:
!pip install transformers torchaudio
!pip install fsspec==2023.9.2
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
from datasets import load_dataset, Audio
import os

dataset = load_dataset("PolyAI/minds14", "en-US", split="train", trust_remote_code=True)
splits = dataset.train_test_split(test_size=0.2)
train_dataset = splits["train"]
test_dataset = splits["test"]

In [ ]:
import torch
import torchaudio
import numpy as np

def resample_audio(sample):
    original_sr = sample["audio"]["sampling_rate"]
    target_sr = 16000
    if original_sr != target_sr:
        waveform = torch.tensor(sample["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        resampler = torchaudio.transforms.Resample(original_sr, target_sr)
        waveform = resampler(waveform)
        sample["audio"]["array"] = waveform.squeeze(0).numpy()
        sample["audio"]["sampling_rate"] = target_sr

    if np.isnan(sample["audio"]["array"]).any() or np.isinf(sample["audio"]["array"]).any():
        print(f"Invalid audio data in sample: {sample['path']}")
        sample["audio"]["array"] = np.zeros_like(sample["audio"]["array"])

    return sample

train_dataset = [resample_audio(sample) for sample in train_dataset]
test_dataset = [resample_audio(sample) for sample in test_dataset]

In [ ]:
import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(train_dataset))

print("Target text:", train_dataset[rand_int]['english_transcription'])
print("Input array shape:", np.asarray(train_dataset[rand_int]["audio"]["array"]).shape)
print("Sampling rate:", train_dataset[rand_int]["audio"]["sampling_rate"])
ipd.Audio(data=np.asarray(train_dataset[rand_int]["audio"]["array"]), autoplay=True, rate=16000)

Target text: hi good morning to you don't like to make a complaint I've been trying to look up my house I don't know why thank you
Input array shape: (159744,)
Sampling rate: 16000


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-base")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]
    input_features = processor.feature_extractor(
        audio["array"], sampling_rate=16000, return_tensors="pt"
    ).input_features[0]
    labels = processor.tokenizer(example["transcription"], return_tensors="pt").input_ids[0]
    return {"input_features": input_features, "labels": labels}

train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
eval_dataset = test_dataset.map(prepare_dataset, remove_columns=test_dataset.column_names)

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/113 [00:00<?, ? examples/s]

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor
    padding: bool = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    return_tensors: str = "pt"

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./whisper-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=10,
    eval_strategy="steps",
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    fp16=True,
    save_total_limit=2,
    report_to="none",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=processor.tokenizer,
)

trainer.train()

<ipython-input-11-d0c4428ab1f5>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=280, training_loss=0.38915637816701615, metrics={'train_runtime': 701.72, 'train_samples_per_second': 6.413, 'train_steps_per_second': 0.399, 'total_flos': 2.8240042328064e+17, 'train_loss': 0.38915637816701615, 'epoch': 9.672566371681416})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.5322319865226746,
 'eval_runtime': 15.351,
 'eval_samples_per_second': 7.361,
 'eval_steps_per_second': 0.977,
 'epoch': 9.672566371681416}

In [ ]:
model.save_pretrained("./whisper-finetuned/final/new")
processor.save_pretrained("./whisper-finetuned/final/new")

[]

In [ ]:
train_dataset.pop("path")
print(train_dataset[0])

TypeError: 'str' object cannot be interpreted as an integer

In [ ]:
import torch
import torchaudio
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Trainer, TrainingArguments
from evaluate import load as load_metric
import numpy as np

# Load the MINDS-14 dataset (English US subset)
dataset = load_dataset("PolyAI/minds14", "en-US", split="train", trust_remote_code=True)

dataset = dataset.filter(lambda x: len(x["transcription"].strip()) > 0)

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base")

split_dataset = dataset.train_test_split(test_size=0.2)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

def resample_audio(batch):
    original_sr = batch["audio"]["sampling_rate"]
    target_sr = 16000
    if original_sr != target_sr:
        waveform = torch.tensor(batch["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        resampler = torchaudio.transforms.Resample(original_sr, target_sr)
        waveform = resampler(waveform)
        batch["audio"]["array"] = waveform.squeeze(0).numpy()
        batch["audio"]["sampling_rate"] = target_sr
    return batch

train_dataset = [resample_audio(sample) for sample in train_dataset]
test_dataset = [resample_audio(sample) for sample in test_dataset]

# Preprocess data
def prepare_dataset(batch):
    audio = batch["audio"]
    # Process audio
    batch["input_values"] = processor(audio["array"], sampling_rate=16000).input_values[0]
    # Process transcription
    transcription = batch["transcription"].lower()
    with processor.as_target_processor():
        batch["labels"] = processor.tokenizer(transcription).input_ids
    return batch

train_dataset = [prepare_dataset(sample) for sample in train_dataset]
test_dataset = [prepare_dataset(sample) for sample in test_dataset]
# Data collator
from dataclasses import dataclass
from typing import Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors="pt")

        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

# Load WER metric
wer_metric = load_metric("wer")

# Compute metrics
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# Training arguments
training_args = TrainingArguments(
    output_dir="./wav2vec2-finetuned-minds14",
    group_by_length=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    num_train_epochs=30,
    fp16=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=500,
    learning_rate=1e-4,
    weight_decay=0.005,
    warmup_steps=1000,
    save_total_limit=2,
    report_to="none",
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=processor.feature_extractor,
)

# Train the model
trainer.train()

# Save the model
trainer.save_model("./model/wav2vec2-finetuned-minds14")
processor.save_pretrained("./model/wav2vec2-finetuned-minds14")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-41-3113677893>:102: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Step,Training Loss,Validation Loss,Wer
500,263.965300,440.890350,1.000000
1000,51.028300,757.367737,1.000000
1500,49.436000,681.081909,1.000000


/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

[]